In [1]:
from pyspark.sql import SparkSession
import getpass

In [2]:
username = getpass.getuser()

In [3]:
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [4]:
sc = spark.sparkContext

In [5]:
orders_base_rdd = sc.textFile("data/retail_db_orders_part-00000")
customers_base_rdd = sc.textFile("data/retail_db_customers_part-00000")
order_items_base_rdd = sc.textFile("data/retail_db_order_items_part-00000")

## The revenue generated by each state in sorted order.

In [6]:
revenue_orders = orders_base_rdd.filter(lambda x: x.split(",")[3] != 'PENDING_PAYMENT')

In [7]:
revenue_orders.take(20)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE',
 '6,2013-07-25 00:00:00.0,7130,COMPLETE',
 '7,2013-07-25 00:00:00.0,4530,COMPLETE',
 '8,2013-07-25 00:00:00.0,2911,PROCESSING',
 '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '14,2013-07-25 00:00:00.0,9842,PROCESSING',
 '15,2013-07-25 00:00:00.0,2568,COMPLETE',
 '17,2013-07-25 00:00:00.0,2667,COMPLETE',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '20,2013-07-25 00:00:00.0,9198,PROCESSING',
 '21,2013-07-25 00:00:00.0,2711,PENDING',
 '22,2013-07-25 00:00:00.0,333,COMPLETE',
 '24,2013-07-25 00:00:00.0,11441,CLOSED',
 '25,2013-07-25 00:00:00.0,9503,CLOSED',
 '26,2013-07-25 00:00:00.0,7562,COMPLETE',
 '28,2013-07-25 00:00:00.0,656,COMPLETE']

In [8]:
orders_cols = orders_base_rdd.map(lambda x: (x.split(",")[0], x.split(",")[2]))

In [11]:
order_items_cols = order_items_base_rdd.map(lambda x: (x.split(",")[1], float(x.split(",")[4])))

In [13]:
order_items_col = order_items_cols.reduceByKey(lambda x, y: x+y)

In [14]:
items_orders_join = orders_cols.join(order_items_col)

In [15]:
# (order_id, (order_customer_id, order_total))
items_orders_join.take(10)

[('4', ('8827', 699.85)),
 ('10', ('5648', 651.9200000000001)),
 ('12', ('1837', 1299.8700000000001)),
 ('16', ('7276', 419.93)),
 ('20', ('9198', 879.8599999999999)),
 ('24', ('11441', 829.97)),
 ('44', ('10500', 399.98)),
 ('50', ('5225', 429.97)),
 ('56', ('10519', 699.89)),
 ('57', ('7073', 637.9))]

In [21]:
items_orders_join_map = items_orders_join.map(lambda x: x[1])

In [22]:
items_orders_join_map.take(10)

[('3066', 429.97),
 ('7733', 1039.88),
 ('1558', 414.94),
 ('7248', 129.99),
 ('12328', 657.9300000000001),
 ('1266', 219.88),
 ('4059', 679.9300000000001),
 ('6488', 1159.89),
 ('1807', 699.96),
 ('1982', 549.85)]

In [16]:
# (customer_id, customer_state)
customers_rdd_cols = customers_base_rdd.map(lambda x: (x.split(",")[0], x.split(",")[7]))

In [19]:
customers_rdd_cols.take(10)

[('1', 'TX'),
 ('2', 'CO'),
 ('3', 'PR'),
 ('4', 'CA'),
 ('5', 'PR'),
 ('6', 'NJ'),
 ('7', 'PR'),
 ('8', 'MA'),
 ('9', 'PR'),
 ('10', 'VA')]

In [23]:
final_join = customers_rdd_cols.join(items_orders_join_map)

In [24]:
final_join.take(10)

[('6252', ('TX', 699.9100000000001)),
 ('6252', ('TX', 609.9000000000001)),
 ('6261', ('CA', 369.97)),
 ('6261', ('CA', 1329.8100000000002)),
 ('6261', ('CA', 129.99)),
 ('6261', ('CA', 199.99)),
 ('6264', ('IL', 399.98)),
 ('6264', ('IL', 1159.9)),
 ('6264', ('IL', 1009.8900000000001)),
 ('6280', ('PR', 1014.96))]

In [26]:
final_agg = final_join.map(lambda x: x[1]).reduceByKey(lambda y,z: y+z).sortBy(lambda x: x[1], ascending = False)

In [27]:
final_agg.take(10)

[('PR', 13208867.689999657),
 ('CA', 5542722.999999963),
 ('NY', 2152706.7399999993),
 ('TX', 1731407.4900000005),
 ('IL', 1457225.830000002),
 ('FL', 1048609.7700000005),
 ('OH', 773804.11),
 ('MI', 730078.97),
 ('PA', 724375.93),
 ('NJ', 606550.99)]

In [28]:
spark.stop()